# Лабораторная работа №2
## Интерполяция, численное дифференцирование и интегрирование

**Группа:** …

В работе нужно реализовать изученные методы и исследовать, как выбор узлов и шага влияет на результат. Готовые примеры помогут увидеть их применение: восстановить траекторию манипулятора, оценить скорость по измеренным координатам и нарисовать фигуру с помощью робота Go2.

В каждом разделе сначала прочитайте условие, затем заполните функции с пометкой `TODO`. Удалите из заполненной функции строку `raise NotImplementedError(...)`: она обозначает ещё не выполненную часть. После этого запустите проверку и ячейки с примерами. Пишите ответы на вопросы в текстовых ячейках под соответствующими графиками.

В разделах А–В реализуйте численные методы самостоятельно. Готовые функции библиотек можно использовать для проверки. В последнем разделе Г, наоборот, нужно выбрать и применить готовую интерполяцию из SciPy. Код анимаций и управления роботами дан; изменять его для выполнения основных заданий не требуется.

Сдайте один заполненный блокнот с кодом, результатами расчётов и выводами. Пункты с пометкой «Дополнительно» выполнять не обязательно.


### Подготовка

Откройте блокнот в Google Colab. Для разделов А–В достаточно стандартных библиотек, отдельная установка не нужна.

Следующая ячейка устанавливает программы для заключительного задания с Go2. Если планируете выполнять его, запустите установку сейчас, затем выберите в Colab **«Среда выполнения → Перезапустить сеанс»** и продолжите с ячейки импортов. Повторять установку после перезапуска не нужно. Если пока выполняете только А–В, пропустите установку.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y libosmesa6
%pip -q install mujoco==3.3.7 pin==3.7.0 "cmeel-urdfdom<5" "cmeel-tinyxml2<11" do-mpc==5.1.2 casadi==3.7.2 "numpy>=1.26,<3" scipy matplotlib "imageio[ffmpeg]>=2.37,<3" pillow


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libosmesa6:amd64.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../libosmesa6_25.1.7-1ubuntu2~24.04.2_amd64.deb ...
Unpacking libosmesa6:amd64 (25.1.7-1ubuntu2~24.04.2) ...
Setting up libosmesa6:amd64 (25.1.7-1ubuntu2~24.04.2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.9) ...
/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a s

In [ ]:
import os
os.environ["MUJOCO_GL"] = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
for variable in ["OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[variable] = "1"

import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Polygon
from matplotlib.collections import LineCollection
from IPython.display import HTML, display
from scipy.interpolate import CubicSpline
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter

np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})


## А. Интерполяция
### А1. Реализация методов

Даны узлы `x` и значения функции в них `y`. Постройте полином Лагранжа и кубический сплайн, чтобы вычислять приближённые значения между узлами.

Заполните две функции:

- `lagrange_interpolant(x, y)` — построение полинома Лагранжа. Можно использовать обычную или барицентрическую форму.
- `cubic_spline(x, y, bc, slopes)` — построение кубического сплайна. При `bc="natural"` вторые производные на концах равны нулю. При `bc="clamped"` первые производные на концах заданы в `slopes=(left, right)`. Второй вариант понадобится для примера с мячом, где известны начальная и конечная скорости.

Результатом каждой функции должна быть другая функция, которая вычисляет приближение. Например:

```python
approximation = lagrange_interpolant(x, y)
value = approximation(0.5)               # значение в одной точке
values = approximation(np.linspace(x[0], x[-1], 100))
```

Начните с обычной функции одной переменной: `x` и `y` — одномерные массивы одинаковой длины. Узлы `x` расположены по возрастанию. При запросе нескольких точек возвращайте массив значений той же длины.

Затем добавьте поддержку траектории: в этом случае `x` содержит время, а `y` — таблицу из двух столбцов с координатами. Интерполируйте каждый столбец отдельно. Для `m` моментов времени результатом будет таблица из `m` строк и двух столбцов. При заданных производных `left` и `right` также содержат по две компоненты.

Функции `prepare_nodes` и `evaluation_points` проверяют данные, а `tridiagonal_solve` решает трёхдиагональную СЛАУ. Их реализация уже дана. Значения за пределами промежутка между первым и последним узлами вычислять не требуется.

После реализации запустите следующую проверку. Она сравнит сплайн с SciPy и проверит совпадение с исходными значениями в узлах. Прохождение проверки не заменяет исследование в следующих пунктах.


In [ ]:
def prepare_nodes(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.ndim != 1 or len(x) < 2:
        raise ValueError("Требуется не менее двух узлов")
    if y.ndim not in (1, 2) or y.shape[0] != len(x):
        raise ValueError("Несогласованные размеры x и y")
    if not np.all(np.isfinite(x)) or not np.all(np.isfinite(y)):
        raise ValueError("Данные должны быть конечными")
    if np.any(np.diff(x) <= 0):
        raise ValueError("Узлы должны строго возрастать")
    return x, y


def evaluation_points(points, x):
    points = np.asarray(points, dtype=float)
    if not np.all(np.isfinite(points)):
        raise ValueError("Точки должны быть конечными")
    if np.any((points < x[0]) | (points > x[-1])):
        raise ValueError("Экстраполяция не предусмотрена")
    return points


def lagrange_interpolant(x, y):
    """Вернуть функцию, вычисляющую полином Лагранжа."""
    # TODO А1: реализовать вычисление полинома Лагранжа в выбранной форме.
    # Возвращаемая функция должна отдельно обрабатывать совпадение с узлом.
    # Используйте prepare_nodes и evaluation_points для проверки данных.
    raise NotImplementedError("А1: интерполяция Лагранжа")

def tridiagonal_solve(lower, diagonal, upper, rhs):
    diagonal = np.array(diagonal, dtype=float, copy=True)
    rhs = np.array(rhs, dtype=float, copy=True)
    for i in range(1, len(diagonal)):
        if diagonal[i-1] == 0:
            raise np.linalg.LinAlgError("Нулевой ведущий элемент")
        factor = lower[i-1] / diagonal[i-1]
        diagonal[i] -= factor * upper[i-1]
        rhs[i] -= factor * rhs[i-1]
    if np.any(diagonal == 0):
        raise np.linalg.LinAlgError("Вырожденная система")
    result = np.empty_like(rhs)
    result[-1] = rhs[-1] / diagonal[-1]
    for i in range(len(diagonal)-2, -1, -1):
        result[i] = (rhs[i] - upper[i]*result[i+1]) / diagonal[i]
    return result


def cubic_spline(x, y, bc="natural", slopes=None):
    """Вернуть функцию, вычисляющую кубический сплайн."""
    # TODO А1: собрать систему для коэффициентов или вторых производных.
    # Учесть естественные условия и заданные первые производные на концах.
    # Вернуть функцию вычисления на нужном интервале между узлами.
    raise NotImplementedError("А1: кубический сплайн")


In [ ]:
# Проверка А1.
rng = np.random.default_rng(42)
x_check = np.array([-1., -0.7, 0.1, 0.4, 1.])
y_check = rng.normal(size=(len(x_check), 2))
t_check = np.linspace(-1, 1, 301)
for bc in ("natural", "clamped"):
    slopes_check = (np.array([0.3, -0.2]), np.array([-0.1, 0.4]))
    ours = cubic_spline(x_check, y_check, bc, slopes_check)
    scipy_bc = "natural" if bc == "natural" else (
        (1, slopes_check[0]), (1, slopes_check[1]))
    reference = CubicSpline(x_check, y_check, bc_type=scipy_bc)
    np.testing.assert_allclose(ours(t_check), reference(t_check), atol=1e-11)
    np.testing.assert_allclose(ours(x_check), y_check, atol=1e-12)

polynomial = lambda x: 2*x**3 - x + 1
p = lagrange_interpolant(x_check, polynomial(x_check))
np.testing.assert_allclose(p(t_check), polynomial(t_check), atol=1e-11)
np.testing.assert_allclose(lagrange_interpolant(x_check, y_check)(x_check),
                           y_check, atol=1e-12)
# Отдельно проверяем случай двух узлов.
for bc in ("natural", "clamped"):
    ours = cubic_spline([0, 1], [1, 2], bc, (0., 0.))
    reference = CubicSpline([0, 1], [1, 2], bc_type=bc)
    np.testing.assert_allclose(ours(np.linspace(0, 1, 20)),
                               reference(np.linspace(0, 1, 20)), atol=1e-12)
print("Проверки пройдены.")


### А2. Как число и расположение узлов влияют на ошибку

Исследуйте две функции: $\sin x$ на $[-\pi,\pi]$ и $1/(1+25x^2)$ на $[-1,1]$.

1. Запустите готовый код ниже. Для каждого метода он использует 5, 9, 17 и 33 узла двух видов: равномерные и чебышёвские. Функции построения обоих наборов узлов даны.
2. Рассмотрите график максимальной ошибки. Она оценивается по 4001 проверочной точке: $E=\max_j|\widetilde f(x_j)-f(x_j)|$. Проверочные точки одинаковы для всех методов.
3. По графикам ответьте: всегда ли добавление узлов уменьшает ошибку? Зависит ли ответ от функции и расположения узлов?
4. Найдите пример, в котором интерполянт проходит через заданные узлы, но заметно отклоняется от функции между ними. Покажите его на графике и объясните, почему совпадения в узлах недостаточно для оценки качества.

В выводе укажите, какой способ расположения узлов вы выбрали бы для каждой функции, и обоснуйте выбор результатами расчётов.


In [ ]:
#@title Графики А2
def uniform_nodes(a, b, n):
    return np.linspace(a, b, n)


def chebyshev_nodes(a, b, n):
    # Узлы Чебышёва–Лобатто: обе границы входят в набор.
    z = -np.cos(np.pi*np.arange(n)/(n-1))
    return (a+b)/2 + (b-a)*z/2


cases = [
    ("sin(x)", np.sin, -np.pi, np.pi),
    ("1/(1+25x²)", lambda x: 1/(1+25*x*x), -1., 1.),
]
counts = [5, 9, 17, 33]
node_types = {"Равномерные": uniform_nodes, "Чебышёвские": chebyshev_nodes}
errors = {}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (title, fun, a, b) in zip(axes, cases):
    test = np.linspace(a, b, 4001)
    exact = fun(test)
    for node_name, node_function in node_types.items():
        for method in ("Лагранж", "Сплайн"):
            series = []
            for count in counts:
                nodes = node_function(a, b, count)
                builder = lagrange_interpolant if method == "Лагранж" else cubic_spline
                approximate = builder(nodes, fun(nodes))(test)
                series.append(np.max(np.abs(approximate-exact)))
            errors[title, node_name, method] = series
            style = "-o" if method == "Лагранж" else "--s"
            ax.semilogy(counts, np.maximum(series, 1e-16), style,
                        label=f"{method}: {node_name.lower()}")
    ax.set(title=title, xlabel="Число узлов", ylabel="Максимальная ошибка")
    ax.set_xticks(counts)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"{'Функция':15s} {'Узлы':14s} {'Метод':8s}   n=5       n=9       n=17      n=33")
for (title, node_name, method), series in errors.items():
    print(f"{title:15s} {node_name:14s} {method:8s} " +
          " ".join(f"{error:9.2e}" for error in series))

fun = lambda x: 1/(1+25*x*x)
test = np.linspace(-1, 1, 4001)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (node_name, node_function) in zip(axes, node_types.items()):
    nodes = node_function(-1, 1, 17)
    ax.plot(test, fun(test), "k", label="Исходная функция")
    ax.plot(test, lagrange_interpolant(nodes, fun(nodes))(test), label="Лагранж")
    ax.plot(test, cubic_spline(nodes, fun(nodes))(test), "--", label="Сплайн")
    ax.scatter(nodes, fun(nodes), s=16, c="black", zorder=3)
    ax.set(title=node_name, xlabel="x", ylabel="f(x)")
    ax.legend()
plt.tight_layout()
plt.show()


### А3. Восстановление траекторий

**Полёт мяча.** Положение брошеного мяча измерили в три момента времени. В примере известна также точная траектория, поэтому можно проверить восстановление между измерениями. Следующая ячейка строит полином Лагранжа, естественный сплайн и сплайн с заданными скоростями на концах.

Запустите ячейку. Сравните траектории и ошибки. Какой метод лучше восстановил полёт? Чем объясняется результат? Почему сведения о скоростях могут изменить сплайн? Рассматривайте только промежуток между первым и последним измерением.

**Манипулятор.** Нужно провести конец манипулятора через заданные положения в заданные моменты времени. В массиве `base_t` записано время в секундах, в `base_p` — соответствующие координаты в метрах. Строка `[0.65, 0.35]` означает положение конца: 0,65 м по горизонтали и 0,35 м по вертикали.

1. Запустите пример. Сравните траектории восстановленные по полиному Лагранжа и сплайна сначала для 8 точек, затем после добавления трёх близких положений из `nearby_p`. Где добавление точек сильнее изменило каждую кривую?
2. Сдвиньте одну из добавленных точек на 0,01 м по одной координате и повторите расчёт. Остальные точки и моменты времени оставьте прежними. Сравните изменение кривой рядом с этой точкой и вдали от неё.
3. Замените координаты в `base_p` своими и повторите пример. Проверьте, может ли манипулятор пройти по всей полученной кривой: обе длины звеньев равны 0,7 м. При необходимости измените точки.

Анимация показывает геометрию движения. Силы и ограничения приводов в этом примере не учитываются.


In [ ]:
#@title Полёт мяча: данные и графики
g = 9.81
flight_end = 0.65

def ball_position(t):
    t = np.asarray(t)
    return np.stack([0.15 + 1.5*t, 0.4 + 3.2*t - g*t*t/2], axis=-1)


def ball_velocity(t):
    t = np.asarray(t)
    return np.stack([np.full_like(t, 1.5), 3.2-g*t], axis=-1)


measured_t = np.array([0., 0.3, flight_end])
measured_p = ball_position(measured_t)
flight_t = np.linspace(0, flight_end, 501)
flight_exact = ball_position(flight_t)
flight_models = {
    "Лагранж": lagrange_interpolant(measured_t, measured_p),
    "Естественный сплайн": cubic_spline(measured_t, measured_p),
    "Сплайн с заданными скоростями": cubic_spline(
        measured_t, measured_p, bc="clamped",
        slopes=(ball_velocity(0.), ball_velocity(flight_end))),
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(flight_exact[:, 0], flight_exact[:, 1], "k", lw=3, label="Точный полёт")
for name, interpolant in flight_models.items():
    predicted = interpolant(flight_t)
    error = np.linalg.norm(predicted-flight_exact, axis=1)
    axes[0].plot(predicted[:, 0], predicted[:, 1], "--", label=name)
    axes[1].semilogy(flight_t, np.maximum(error, 1e-16), label=name)
    print(f"{name}: максимальная ошибка {error.max():.3e} м")
axes[0].scatter(measured_p[:, 0], measured_p[:, 1], c="black", zorder=5)
axes[0].set(xlabel="x, м", ylabel="z, м", title="Траектория")
axes[0].axis("equal")
axes[1].set(xlabel="t, с", ylabel="Ошибка положения, м")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

meeting_t = 0.48
print("\nПоложение мяча в момент встречи t = 0.48 с:")
print("Точное:", ball_position(meeting_t))
for name, interpolant in flight_models.items():
    print(f"{name}:", interpolant(meeting_t))


In [ ]:
#@title Манипулятор: данные, графики и анимация
base_t = np.array([0., 1., 2., 3., 7., 8., 9., 10.])
base_p = np.array([
    [0.65, 0.35], [0.95, 0.55], [0.85, 0.80], [0.50, 0.65],
    [0.45, 0.30], [0.80, 0.20], [1.00, 0.45], [0.70, 0.35],
])

# На участке между t=3 и t=7 добавляем небольшой изгиб.
nearby_t = np.array([4., 5., 6.])
nearby_p = np.array([[0.48, 0.66], [0.50, 0.63], [0.52, 0.65]])
order = np.argsort(np.r_[base_t, nearby_t])
key_t = np.r_[base_t, nearby_t][order]
key_p = np.vstack([base_p, nearby_p])[order]

arm_l1 = arm_l2 = 0.7
arm_t = np.linspace(key_t[0], key_t[-1], 1001)
colors = {"Лагранж": "tab:orange", "Сплайн": "tab:blue"}
base_models = {
    "Лагранж": lagrange_interpolant(base_t, base_p),
    "Сплайн": cubic_spline(base_t, base_p),
}
arm_models = {
    "Лагранж": lagrange_interpolant(key_t, key_p),
    "Сплайн": cubic_spline(key_t, key_p),
}
arm_paths = {name: fun(arm_t) for name, fun in arm_models.items()}

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, models, title in zip(axes, [base_models, arm_models],
        ["8 исходных точек", "11 точек: добавлен небольшой изгиб"]):
    ax.add_patch(plt.Circle((0, 0), arm_l1+arm_l2,
                           fill=False, color="gray", linestyle=":",
                           label="Граница достижимости"))
    for name, fun in models.items():
        path = fun(arm_t)
        ax.plot(path[:, 0], path[:, 1], color=colors[name], lw=2, label=name)
    ax.scatter(base_p[:, 0], base_p[:, 1], c="black", s=30, zorder=5,
               label="Исходные точки")
    ax.scatter(0, 0, c="black", marker="s", s=55)
    if models is arm_models:
        ax.scatter(nearby_p[:, 0], nearby_p[:, 1], c="deeppink", s=40,
                   zorder=6, label="Добавленные точки")
    ax.set(title=title, xlabel="x, м", ylabel="z, м",
           xlim=(-0.2, 1.45), ylim=(-0.9, 1.05))
    ax.set_aspect("equal")
    ax.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

# Подробно показываем участок с новыми точками.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, path in arm_paths.items():
    axes[0].plot(path[:, 0], path[:, 1], color=colors[name], lw=2, label=name)
    for j in range(2):
        axes[j+1].plot(arm_t, path[:, j], color=colors[name], label=name)
axes[0].scatter(key_p[:, 0], key_p[:, 1], c="black", s=18, zorder=4)
axes[0].scatter(nearby_p[:, 0], nearby_p[:, 1], c="deeppink", s=40, zorder=5)
for t, point in zip(nearby_t, nearby_p):
    axes[0].annotate(f"t={t:g}", point, xytext=(6, 5), textcoords="offset points")
axes[0].set(xlabel="x, м", ylabel="z, м", title="Участок с добавленными точками",
            xlim=(0.43, 0.57), ylim=(0.58, 0.70))
axes[0].set_aspect("equal")
for j, coordinate in enumerate(["x", "z"]):
    axes[j+1].scatter(base_t, base_p[:, j], c="black", s=20, zorder=4)
    axes[j+1].scatter(nearby_t, nearby_p[:, j], c="deeppink", s=35, zorder=5)
    axes[j+1].set(xlabel="t, с", ylabel=f"{coordinate}, м")
    axes[j+1].axvspan(4, 6, color="deeppink", alpha=0.07)
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Проверка траекторий после добавления точек:")
for name, path in arm_paths.items():
    radius = np.linalg.norm(path, axis=1)
    unreachable = radius > arm_l1+arm_l2+1e-12
    displacement = np.linalg.norm(path-base_models[name](arm_t), axis=1)
    np.testing.assert_allclose(arm_models[name](key_t), key_p, atol=1e-10)
    print(f"{name}: наибольшее изменение траектории {displacement.max():.3f} м; "
          f"максимальное расстояние до основания {radius.max():.3f} м; "
          f"недостижимых отсчётов {np.count_nonzero(unreachable)}/{len(arm_t)}")

# Дополнительный опыт: сдвигаем одну из новых точек на 1 см.
changed_p = key_p.copy()
changed_index = int(np.where(key_t == 5.)[0][0])
changed_p[changed_index, 1] += 0.01
changed_models = {
    "Лагранж": lagrange_interpolant(key_t, changed_p),
    "Сплайн": cubic_spline(key_t, changed_p),
}
plt.figure(figsize=(8, 4))
for name, fun in changed_models.items():
    change = np.linalg.norm(fun(arm_t)-arm_paths[name], axis=1)
    plt.plot(arm_t, change, color=colors[name], label=name)
plt.axvline(key_t[changed_index], color="black", linestyle=":", label="Изменённый кадр")
plt.xlabel("t, с")
plt.ylabel("Изменение положения, м")
plt.legend()
plt.show()

def arm_configuration(point):
    x, z = point
    cosine = (x*x + z*z-arm_l1**2-arm_l2**2)/(2*arm_l1*arm_l2)
    if cosine < -1-1e-12 or cosine > 1+1e-12:
        return None
    q2 = np.arccos(np.clip(cosine, -1, 1))
    q1 = np.arctan2(z, x)-np.arctan2(arm_l2*np.sin(q2), arm_l1+arm_l2*np.cos(q2))
    elbow = np.array([arm_l1*np.cos(q1), arm_l1*np.sin(q1)])
    return np.array([[0., 0.], elbow, point])


fig, axes = plt.subplots(1, 2, figsize=(10, 4))
artists = {}
for ax, (name, path) in zip(axes, arm_paths.items()):
    ax.plot(path[:, 0], path[:, 1], color=colors[name], alpha=0.3)
    ax.scatter(base_p[:, 0], base_p[:, 1], c="black", s=20)
    ax.scatter(nearby_p[:, 0], nearby_p[:, 1], c="deeppink", s=35, zorder=5)
    ax.add_patch(plt.Circle((0, 0), arm_l1+arm_l2,
                           fill=False, color="gray", linestyle=":"))
    ax.scatter(0, 0, c="black", s=40)
    line, = ax.plot([], [], "o-", lw=4, color=colors[name])
    trail, = ax.plot([], [], color=colors[name], lw=1.5)
    target, = ax.plot([], [], "rx", ms=7)
    label = ax.text(0.03, 0.94, "", transform=ax.transAxes)
    artists[name] = line, trail, target, label
    ax.set(title=name, xlabel="x, м", ylabel="z, м", xlim=(-0.2, 1.45), ylim=(-0.9, 1.05))
    ax.set_aspect("equal")


def update_arm(frame):
    for name, path in arm_paths.items():
        line, trail, target, label = artists[name]
        configuration = arm_configuration(path[frame])
        if configuration is None:
            line.set_data([], [])
            target.set_data([path[frame, 0]], [path[frame, 1]])
            label.set_text("Цель недостижима")
        else:
            line.set_data(configuration[:, 0], configuration[:, 1])
            target.set_data([], [])
            label.set_text(f"t = {arm_t[frame]:.2f} с")
        trail.set_data(path[:frame+1, 0], path[:frame+1, 1])
    return []


animation = FuncAnimation(fig, update_arm, frames=np.arange(0, len(arm_t), 10),
                          interval=100, blit=False)
plt.close(fig)
display(HTML(animation.to_jshtml()))


In [ ]:
# TODO А3: собственные ключевые кадры и повторный запуск визуализации.


**Выводы А.** Приведите по одному наблюдению о влиянии узлов, граничных условий сплайна и изменения одной точки траектории. Подкрепите каждое наблюдение графиком или значением ошибки.

Ваш ответ: …


## Б. Численное дифференцирование
### Б1. Реализация формул

Составьте или найдите в литературе четыре разностные формулы. Если используете готовую формулу, укажите источник. Запишите формулы и их порядки точности в текстовой ячейке.

| Обозначение в коде | Что вычислить | Какие значения использовать |
|---|---|---|
| `D1-L4` | $f'(x)$ | $f(x),f(x+h),f(x+2h),f(x+3h)$ |
| `D1-C3` | $f'(x)$ | $f(x-h),f(x),f(x+h)$ |
| `D1-C5` | $f'(x)$ | $f(x-2h),f(x-h),f(x),f(x+h),f(x+2h)$ |
| `D2-C3` | $f''(x)$ | $f(x-h),f(x),f(x+h)$ |

Во всех случаях производная оценивается в точке $x$, шаг $h>0$.

Заполните `diff_at(fun, x, h, scheme)`: аргумент `fun` — функция, а `scheme` — одно из обозначений таблицы. Например, `diff_at(np.sin, 0.7, 0.01, "D1-C3")` должно оценить производную синуса в точке 0,7 по трём точкам с шагом 0,01.

Затем заполните `diff_samples(values, dt, scheme)`. Здесь даны уже вычисленные значения через одинаковое время `dt`. Примените ту же формулу везде, где хватает исходных точек. Верните два массива: номера точек, к которым относятся оценки, и сами оценки. Например, для пяти значений и `D1-C3` это номера `[1, 2, 3]`: в первой и последней точках не хватает соседа. Не подставляйте на границах нулевые производные.

Сначала реализуйте одномерный массив значений. Затем поддержите таблицу координат из двух столбцов, применяя формулу к каждому столбцу отдельно. Это понадобится для оценки скорости и ускорения.


In [ ]:
# Расположение точек и порядок производной заданы условием.
DIFF_STENCILS = {
    "D1-L4": (np.array([0, 1, 2, 3]), 1),
    "D1-C3": (np.array([-1, 0, 1]), 1),
    "D1-C5": (np.array([-2, -1, 0, 1, 2]), 1),
    "D2-C3": (np.array([-1, 0, 1]), 2),
}


def diff_at(fun, x, h, scheme):
    # TODO Б1: применить формулу по имени scheme в точке x.
    # Проверить положительность h и допустимость имени формулы.
    raise NotImplementedError("Б1: производная функции")


def diff_samples(values, dt, scheme):
    # TODO Б1: применить формулу к массиву данных.
    # Вернуть индексы, к которым относятся оценки, и массив производных.
    raise NotImplementedError("Б1: производная по отсчётам")


In [ ]:
# Проверка интерфейса и временного положения результата.
test_t = np.arange(11)*0.1
for name, (_, derivative) in DIFF_STENCILS.items():
    indices, result = diff_samples(test_t**2, 0.1, name)
    exact = 2*test_t[indices] if derivative == 1 else np.full(len(indices), 2.)
    np.testing.assert_allclose(result, exact, atol=1e-11)
print("Проверки пройдены.")


### Б2. Выбор шага

Исследуйте $e^x$, $\sin x$ и $x^3-2x^2+0.5x+1$ в точке $x=0.7$.

1. Запустите готовый код. Он вычисляет производные при шагах от $1$ до $10^{-16}$ и сравнивает их с аналитическими значениями. Ошибка равна модулю разности численной и аналитической производных.
2. Для каждой формулы найдите шаг с наименьшей ошибкой среди проверенных. Всегда ли уменьшение шага полезно? Объясните поведение при очень малых шагах.
3. Сравните результаты для трёх функций. Есть ли случаи, заметно отличающиеся от остальных? Объясните их с помощью полученных формул, не ограничиваясь видом графика.
4. Измените `x_diff` и повторите опыт. Сохранились ли наблюдения?
5. Для $e^x$ сравните ошибки при шагах $h$ и $h/2$. Оцените порядок как $\log_2(E_h/E_{h/2})$ и сопоставьте с порядком вашей формулы. Используйте участок, где ошибка ещё уменьшается с шагом.

На логарифмическом графике ноль показать нельзя. Поэтому в готовом коде нулевые ошибки отмечены отдельно; их положение по вертикали не является величиной ошибки.


In [ ]:
#@title Графики ошибок и наблюдаемые порядки
diff_functions = [
    ("f₁(x) = exp(x)", np.exp, np.exp, np.exp),
    ("f₂(x) = sin(x)", np.sin, np.cos, lambda x: -np.sin(x)),
    ("f₃(x) = x³ − 2x² + 0.5x + 1",
     lambda x: x**3-2*x*x+0.5*x+1,
     lambda x: 3*x*x-4*x+0.5,
     lambda x: 6*x-4),
]
x_diff = 0.7
h_diff = np.logspace(0, -16, 321)
error_by_function = {}
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (label, fun, d1, d2) in zip(axes, diff_functions):
    for scheme in ("D1-L4", "D1-C3", "D1-C5"):
        error = np.array([abs(diff_at(fun, x_diff, h, scheme)-d1(x_diff))
                          for h in h_diff])
        error_by_function[label, scheme] = error
        # Только для логарифмического графика заменяем нули нижней отметкой.
        ax.loglog(h_diff, np.maximum(error, 1e-18), label=scheme)
        zero = error == 0
        if np.any(zero):
            ax.scatter(h_diff[zero], np.full(zero.sum(), 1e-18), marker="v", s=12)
        best = np.argmin(error)
        print(f"{label}; {scheme}: h={h_diff[best]:.2e}, ошибка={error[best]:.2e}")
    ax.set(title=label, xlabel="h", ylabel="Абсолютная ошибка")
    ax.legend()
plt.tight_layout()
plt.show()
print("Нулевая вычисленная ошибка отмечена треугольником на уровне 10⁻¹⁸.")

# Отдельно исследуем вторую производную.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (label, fun, d1, d2) in zip(axes, diff_functions):
    error = np.array([abs(diff_at(fun, x_diff, h, "D2-C3")-d2(x_diff))
                      for h in h_diff])
    ax.loglog(h_diff, np.maximum(error, 1e-18), label="D2-C3")
    ax.set(title=label, xlabel="h", ylabel="Абсолютная ошибка")
    ax.legend()
plt.tight_layout()
plt.show()

# Проверяем, различимы ли соседние аргументы в float64.
print("Шаг числа float64 около x=0.7:", np.spacing(x_diff))
for h in [1e-8, 1e-12, 1e-16, 1e-17]:
    print(f"h={h:.0e}: (x+h)-x = {(x_diff+h)-x_diff:.3e}")

order_steps = 0.1/2**np.arange(6)
print("Наблюдаемые порядки для exp(x):")
for scheme, (_, derivative) in DIFF_STENCILS.items():
    errors = np.array([abs(diff_at(np.exp, x_diff, h, scheme)-np.exp(x_diff))
                       for h in order_steps])
    orders = np.log2(errors[:-1]/errors[1:])
    print(f"{scheme}: {orders}")


### Б3. Скорость и ускорение по измеренной траектории

Автомобиль движется по трассе. Лидар измеряет расстояния до окружающих стен. Сравнив два последовательных набора измерений, можно оценить перемещение и поворот автомобиля. Повторяя эту процедуру, получают его траекторию — это называется лидарной одометрией.

Для сопоставления измерений в подготовленном примере использован метод ICP: он подбирает поворот и перенос, при которых соседние наборы точек лучше совпадают. Изучать или реализовывать ICP в этой работе не требуется. Подробнее о нём можно прочитать в [описании метода](https://www.open3d.org/docs/release/tutorial/pipelines/icp_registration.html).

Вам даны три готовые записи одометрии с разным шумом измерения расстояний: 0,5; 1,5 и 3 см. Координаты записаны каждые 0,1 с. Из-за накопления ошибок даже достаточно правдоподобная траектория отличается от истинной. Нужно выяснить, насколько хорошо по ней можно оценить скорость и ускорение.

Сначала запустите две следующие ячейки и посмотрите анимации. Синий автомобиль показывает истинное положение, оранжевый контур — оценку по лидару. Нижние графики показывают ошибку положения. Начальное положение известно; дополнительные датчики для исправления траектории не используются.


In [ ]:
#@title Загрузить готовые записи и функции визуализации
import base64
import io

_DATA = (
    '...'
    'ABoAGgAABgAAuL4DAAAA'
)
with np.load(io.BytesIO(base64.b64decode(_DATA)), allow_pickle=False) as archive:
    trajectory_reference = {
        "t": archive["t"].copy(),
        "position": archive["true_p"].copy(),
        "velocity": archive["true_v"].copy(),
        "acceleration": archive["true_a"].copy(),
        "heading": archive["true_heading"].copy(),
    }
    trajectory_records = {}
    for j, sigma in enumerate(archive["sigma_range"]):
        label = f"Шум {100*sigma:g} см"
        trajectory_records[label] = {
            "t": archive["t"].copy(),
            "position": archive[f"odom_p_{j}"].copy(),
            "heading": archive[f"odom_heading_{j}"].copy(),
            "fit_errors": archive[f"fit_errors_{j}"].copy(),
            "scan_visual": archive[f"scan_visual_{j}"].copy(),
            "sigma_range": float(sigma),
        }
    trajectory_world = [archive[f"world_{j}"].copy() for j in range(6)]
del _DATA


def rotation(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, -s], [s, c]])


def car_polygon(position, heading):
    corners = np.array([[-0.28, -0.15], [0.28, -0.15],
                        [0.28, 0.15], [-0.28, 0.15]])
    return corners @ rotation(heading).T + position


def vector_rmse(estimated, exact):
    return np.sqrt(np.mean(np.sum((estimated-exact)**2, axis=-1)))


record_names = list(trajectory_records)
print("Готовы три записи; отсчётов в каждой:", len(trajectory_reference["t"]))


In [ ]:
#@title Три одометрии рядом
t_record = trajectory_reference["t"]
p_record_true = trajectory_reference["position"]
fig, axes = plt.subplots(2, 3, figsize=(13, 6), dpi=75,
                          gridspec_kw={"height_ratios": [3, 1]})
comparison_artists = {}
position_errors = {}
max_position_error = max(np.max(np.linalg.norm(r["position"]-p_record_true, axis=1))
                         for r in trajectory_records.values())

for column, (name, record) in enumerate(trajectory_records.items()):
    map_ax, error_ax = axes[:, column]
    for polygon in trajectory_world:
        closed = np.vstack([polygon, polygon[0]])
        map_ax.plot(closed[:, 0], closed[:, 1], color="0.45", lw=1)
    map_ax.plot(p_record_true[:, 0], p_record_true[:, 1], "k--", alpha=0.4,
                label="Истинный путь")
    odom_line, = map_ax.plot([], [], color="tab:orange", lw=2, label="Одометрия")
    true_car = Polygon(car_polygon(p_record_true[0], trajectory_reference["heading"][0]),
                       facecolor="tab:blue", zorder=5)
    estimated_car = Polygon(car_polygon(record["position"][0], record["heading"][0]),
                            fill=False, edgecolor="tab:orange", lw=2, zorder=6)
    map_ax.add_patch(true_car)
    map_ax.add_patch(estimated_car)
    rays = LineCollection([], colors="seagreen", linewidths=0.5, alpha=0.35)
    map_ax.add_collection(rays)
    map_ax.set(title=name, xlim=(-9, 9), ylim=(-7, 7), xlabel="x, м", ylabel="y, м")
    map_ax.set_aspect("equal")
    map_ax.legend(loc="lower left", fontsize=7)
    error = np.linalg.norm(record["position"]-p_record_true, axis=1)
    position_errors[name] = error
    error_ax.plot(t_record, error, color="tab:orange", lw=1)
    cursor = error_ax.axvline(0, color="tab:blue")
    error_ax.set(xlabel="t, с", ylabel="Ошибка, м", ylim=(0, 1.05*max_position_error))
    comparison_artists[name] = (true_car, estimated_car, rays, odom_line, cursor)
comparison_clock = fig.suptitle("t = 0.0 с")
plt.tight_layout()


def update_comparison(i):
    position = p_record_true[i]
    heading = trajectory_reference["heading"][i]
    for name, record in trajectory_records.items():
        true_car, estimated_car, rays, odom_line, cursor = comparison_artists[name]
        true_car.set_xy(car_polygon(position, heading))
        estimated_car.set_xy(car_polygon(record["position"][i], record["heading"][i]))
        local = record["scan_visual"][i]
        local = local[np.all(np.isfinite(local), axis=1)]
        cloud_world = local @ rotation(heading).T + position
        starts = np.broadcast_to(position, cloud_world.shape)
        rays.set_segments(np.stack([starts, cloud_world], axis=1))
        odom_line.set_data(record["position"][:i+1, 0], record["position"][:i+1, 1])
        cursor.set_xdata([t_record[i], t_record[i]])
    comparison_clock.set_text(f"t = {t_record[i]:.1f} с")
    return []


comparison_frames = np.unique(np.r_[np.arange(0, len(t_record), 5), len(t_record)-1])
comparison_animation = FuncAnimation(fig, update_comparison, frames=comparison_frames,
                                      interval=500, blit=False)
plt.close(fig)
display(HTML(comparison_animation.to_jshtml()))


### Работа с данными

Каждая запись содержит время и две координаты положения. Например:

```python
name = record_names[0]                       # первая запись
t = trajectory_records[name]["t"]           # время, с
position = trajectory_records[name]["position"]  # столбцы x и y, м
```

Истинные положение, скорость и ускорение находятся в `trajectory_reference` под ключами `"position"`, `"velocity"` и `"acceleration"`. Их строки относятся к тем же моментам времени. Они нужны для проверки, а не для вычисления производных измеренной траектории.

1. Для первой записи оцените обе компоненты скорости тремя формулами первой производной. Формулой второй производной оцените обе компоненты ускорения. Постройте их графики вместе с истинными значениями. Скорость измеряется в м/с, ускорение — в м/с².
2. Оцените ошибку каждого результата. Если `indices` — номера точек, возвращённые `diff_samples`, сравнивайте оценки с истинными значениями именно в этих точках. Используйте
   $$\mathrm{RMSE}=\sqrt{\frac1N\sum_{i=1}^N\left[(\widehat w_{x,i}-w_{x,i})^2+(\widehat w_{y,i}-w_{y,i})^2\right]},$$
   где $w$ — скорость или ускорение. Так сравниваются сами векторы, а не только их длины.
3. Повторите расчёт, оставляя каждую вторую, третью, пятую и десятую точку. Например, `position[::5]` — координаты с шагом 0,5 с. Проредите таким же способом время и истинные значения. Новую одометрию вычислять не нужно. Для каждого шага используйте при сравнении формул одни и те же моменты времени в пределах от 2 до 27 с, где определены все сравниваемые оценки.
4. Повторите опыт для остальных двух записей. Постройте зависимость ошибки от шага или составьте таблицу. Как влияет шум? Помогает ли увеличение числа точек в формуле? Какой шаг оказался удачным для скорости и для ускорения?
5. Вычислите производные по **истинным координатам** теми же формулами и с теми же шагами. Сравните ошибки: какая их часть связана с численным методом, а какая — с неточными координатами?

**Дополнительно.** Выберите метод и шаг по результатам вне промежутка 10–20 с, затем проверьте их на этом промежутке. Можно также сравнить прореживание с предварительным сглаживанием координат. Укажите способ сглаживания и его параметры.


In [ ]:
selected_record = record_names[-1]
t_measurements = trajectory_records[selected_record]["t"].copy()
p_measurements = trajectory_records[selected_record]["position"].copy()
strides = [1, 2, 3, 5, 10]

# TODO Б3: оценки производных, общие временные отсчёты,
# RMSE, графики и сравнение с истинными координатами.


**Выводы Б.** Объясните, почему шаг с наименьшей ошибкой для точной функции и для зашумленной траектории может различаться. Укажите выбранные методы и шаги для оценки скорости и ускорения; приведите полученные ошибки.

Ваш ответ: …


## В. Численное интегрирование
### В1. Реализация методов

Заполните функции для четырёх составных квадратурных формул: левых прямоугольников, средних прямоугольников, трапеций и Симпсона.

Каждая функция получает `fun, a, b, n` и возвращает одно число — приближение интеграла $\int_a^b f(x)\,dx$. Здесь `n` — число равных отрезков разбиения, а не число узлов. Шаг равен $(b-a)/n$. Например, `trapezoids(np.sin, 0, np.pi, 100)` должно приближённо вычислить интеграл синуса на $[0,\pi]$ методом трапеций.

Для формулы Симпсона используйте чётное `n`; при нечётном значении вызывайте `ValueError`. Учтите также перестановку пределов: значение интеграла должно менять знак.

Функции построения сетки и вычисления значений уже даны. Готовые функции интегрирования из библиотек внутри своих реализаций не используйте. После заполнения запустите проверки.


In [ ]:
def quadrature_grid(a, b, n):
    if isinstance(n, (bool, np.bool_)) or not isinstance(n, (int, np.integer)) or n <= 0:
        raise ValueError("n должно быть положительным целым")
    if not np.isfinite(a) or not np.isfinite(b):
        raise ValueError("Границы должны быть конечными")
    return np.linspace(a, b, n+1), (b-a)/n


def function_values(fun, x):
    values = np.broadcast_to(np.asarray(fun(x), dtype=float), x.shape)
    if not np.all(np.isfinite(values)):
        raise ValueError("Функция приняла нечисловое или бесконечное значение")
    return values


def left_rectangles(fun, a, b, n):
    # TODO В1: составная формула левых прямоугольников.
    raise NotImplementedError("В1: левые прямоугольники")


def midpoint_rectangles(fun, a, b, n):
    # TODO В1: составная формула средних прямоугольников.
    raise NotImplementedError("В1: средние прямоугольники")


def trapezoids(fun, a, b, n):
    # TODO В1: составная формула трапеций.
    raise NotImplementedError("В1: трапеции")


def simpson(fun, a, b, n):
    # TODO В1: составная формула Симпсона; проверить чётность n.
    raise NotImplementedError("В1: Симпсон")


# Порядки в этой таблице предполагают достаточную гладкость функции.
QUADRATURES = {
    "Левые прямоугольники": (left_rectangles, 1),
    "Средние прямоугольники": (midpoint_rectangles, 2),
    "Трапеции": (trapezoids, 2),
    "Симпсон": (simpson, 4),
}
quad_colors = dict(zip(QUADRATURES, ["tab:blue", "tab:orange", "tab:green", "tab:red"]))


In [ ]:
# Базовые проверки реализации.
for method, _ in QUADRATURES.values():
    np.testing.assert_allclose(method(lambda x: 3., 0., 2., 8), 6., atol=1e-14)
    np.testing.assert_allclose(method(lambda x: 3., 2., 0., 8), -6., atol=1e-14)
np.testing.assert_allclose(left_rectangles(lambda x: 2*x+1, 0., 1., 4), 1.75)
for method in (midpoint_rectangles, trapezoids, simpson):
    np.testing.assert_allclose(method(lambda x: 2*x+1, 0., 1., 8), 2., atol=1e-14)
try:
    simpson(np.exp, 0., 1., 3)
except ValueError:
    pass
else:
    raise AssertionError("Нечётное n для Симпсона не отклонено")
print("Базовые проверки пройдены.")


### В2. Ошибка и скорость её уменьшения

Для исследования даны три функции с известными интегралами:

| Функция | Промежуток | Точный интеграл |
|---|---|---|
| $e^x$ | $[0,1]$ | $e-1$ |
| $x^3-2x+1$ | $[0,1]$ | $1/4$ |
| $\sqrt{x}$ | $[0,1]$ | $2/3$ |

1. Вычислите каждый интеграл всеми четырьмя методами при $n=2,4,8,\ldots,16384$. Функции, точные значения и набор `n` уже заданы в следующей ячейке.
2. Найдите абсолютную ошибку $E_n=|I_n-I|$ и постройте её зависимость от шага $h=1/n$ в логарифмических масштабах. Подпишите методы.
3. Для нескольких соседних разбиений вычислите $p_n=\log_2(E_n/E_{2n})$. Это оценка того, насколько быстро уменьшается ошибка. Сравните её с теоретическим порядком метода, указанным в `QUADRATURES`.
4. Объясните различия между функциями. Всегда ли наблюдается теоретический порядок? Посмотрите отдельно на полином и на поведение $\sqrt{x}$ около нуля.

Если ошибка получилась нулевой, запишите это в таблице: логарифм нуля вычислять не нужно. При ошибках, сравнимых с погрешностью машинных вычислений, отношение ошибок уже не позволяет надёжно оценивать порядок.


In [ ]:
integration_cases = [
    ("exp(x)", np.exp, 0., 1., float(np.expm1(1.))),
    ("x³ − 2x + 1", lambda x: x**3-2*x+1, 0., 1., 0.25),
    ("√x", np.sqrt, 0., 1., 2/3),
]
interval_counts = 2**np.arange(1, 15)

# TODO В2: приближения, фактические ошибки, графики и наблюдаемые порядки.


### В3. Оценка ошибки по правилу Рунге

В предыдущем пункте ошибка была известна благодаря точному интегралу. Теперь оцените её только по двум численным результатам.

1. Вычислите интегралы $I_n$ и $I_{2n}$ одним методом. Для метода порядка $p$ оценка ошибки результата на более мелком разбиении равна
   $$R_{2n} \approx \frac{|I_{2n}-I_n|}{2^p-1}.$$
2. Заполните `runge_estimate(method, fun, a, b, n, p)`. Она должна вернуть словарь с четырьмя числами: `coarse` — $I_n$, `fine` — $I_{2n}$, `runge` — $R_{2n}$ и `refined` — уточнённое значение
   $$I_{\mathrm{уточн}}=I_{2n}+\frac{I_{2n}-I_n}{2^p-1}.$$
   Для поправки важен знак разности: модуль здесь не берётся.
3. Для трёх функций из В2 сравните $R_{2n}$ с фактической ошибкой $|I_{2n}-I|$ при нескольких разбиениях. Используйте порядки из `QUADRATURES`. Покажите сравнение в таблице или на графике.
4. Сравните ошибку уточнённого значения с ошибкой $I_{2n}$. В каких опытах уточнение помогло? Объясните случаи, где оценка Рунге плохо соответствует фактической ошибке. Проверьте, выполнены ли условия для использования выбранного порядка.

**Дополнительно.** Оцените порядок, не используя точный интеграл: $p\approx\log_2\bigl(|I_{2n}-I_n|/|I_{4n}-I_{2n}|\bigr)$. Не используйте для этого разности, сравнимые с погрешностью машинных вычислений.


In [ ]:
def runge_estimate(method, fun, a, b, n, p):
    """Оценка ошибки на мелкой сетке и уточнение интеграла."""
    # TODO В3: вычислить I_n и I_2n, оценку ошибки и уточнённое значение.
    # Вернуть словарь с ключами:
    # coarse — I_n; fine — I_2n; runge — неотрицательная оценка ошибки I_2n;
    # refined — уточнённое значение с учётом знака поправки.
    raise NotImplementedError("В3: правило Рунге")

# TODO В3: сравнение с фактической ошибкой, таблицы и графики.


**Выводы В.** Для каждой функции укажите, как менялась ошибка при сгущении сетки. Объясните, в каких случаях правилу Рунге можно доверять по результатам вашего исследования.

Ваш ответ: …


## Г. Заключительное задание: рисование с Unitree Go2

В предыдущих заданиях вы интерполировали зависимость от одной переменной. Здесь нужно восстановить зависимость от двух переменных с помощью готовой функции SciPy. Познакомьтесь с подходами к [https://en.wikipedia.org/wiki/Multivariate_interpolation](интерполяции функций многих переменных) и доступными реализациями в [https://docs.scipy.org/doc/scipy/tutorial/interpolate.html](документации SciPy)

Робособака Go2 стоит на месте и поворачивает корпус. Две команды задают его наклон вперёд-назад (`pitch`) и поворот влево-вправо (`yaw`). Перед корпусом закреплена виртуальная метка — «кончик пера». Неподвижная камера видит её в точке экрана с координатами $(u,v)$. Координата $u$ растёт вправо, $v$ — вниз; обе измеряются в пикселях. Команды углов задаются в радианах.

Нужно научиться решать обратную задачу: по желаемой точке экрана находить две команды роботу. Для этого:

1. Подайте несколько известных пар команд и запишите положения метки на экране.
2. По этим данным постройте две зависимости: $\mathrm{pitch}(u,v)$ и $\mathrm{yaw}(u,v)$.
3. Задайте последовательность точек рисунка, получите команды и посмотрите на движение робота.

Симуляция, удержание стойки и измерение положения метки уже реализованы. Разбирать алгоритм управления для этого задания не требуется. Модель скачивается из официального MuJoCo Menagerie; её версия зафиксирована в коде. Если зависимости ещё не установлены, выполните установку из начала блокнота и перезапустите сеанс до запуска следующей ячейки.


In [ ]:
#@title Готовая инфраструктура Go2
import sys
import subprocess
import warnings
import json
from pathlib import Path
import mujoco
from IPython.display import Video
warnings.filterwarnings("ignore", module="do_mpc")
WORK_DIR = Path.cwd() / "go2_interpolation"
WORK_DIR.mkdir(exist_ok=True)
PROJECT_FILES = json.loads('{"go2_sim.py": "\\"\\"\\"Простой Unitree Go2: стойка, фиксированная камера, место для MPC.\\n\\npython go2_sim.py                            # окно на 10 секунд\\npython go2_sim.py --headless --duration 10    # только физика\\nMUJOCO_GL=egl python go2_sim.py --video outputs/go2.mp4\\n\\"\\"\\"\\n\\nimport argparse\\nimport math\\nfrom pathlib import Path\\nimport time\\n\\nimport mujoco\\nimport numpy as np\\n\\n\\nSCENE_PATH = Path(__file__).resolve().parent / \\"scene.xml\\"\\nCAMERA_NAME = \\"fixed\\"\\nCONTROL_SUBSTEPS = 5  # Физика 500 Гц, контроллер 100 Гц (каждые 10 мс).\\nFPS = 30\\nWIDTH, HEIGHT = 960, 544\\n\\n\\ndef load_simulation():\\n    \\"\\"\\"Загрузить сцену и поставить робота в исходную стойку home.\\"\\"\\"\\n    model = mujoco.MjModel.from_xml_path(str(SCENE_PATH))\\n    data = mujoco.MjData(model)\\n    mujoco.mj_resetDataKeyframe(model, data, model.key(\\"home\\").id)\\n    mujoco.mj_forward(model, data)\\n    return model, data\\n\\n\\ndef standing_controller(model, data):\\n    \\"\\"\\"ЗАМЕНИТЬ НА MPC: вернуть 12 моментов двигателей, единицы Н·м.\\n\\n    Здесь только PD для удержания начальной стойки на ровном полу.\\n    Индексы получаем из модели: порядок суставов и приводов может различаться.\\n    \\"\\"\\"\\n    joint_ids = model.actuator_trnid[:, 0]\\n    q_indices = model.jnt_qposadr[joint_ids]\\n    v_indices = model.jnt_dofadr[joint_ids]\\n    q_reference = model.key(\\"home\\").qpos[q_indices]\\n    q = data.qpos[q_indices]\\n    dq = data.qvel[v_indices]\\n    return 60.0 * (q_reference - q) - 3.0 * dq\\n\\n\\ndef step_simulation(model, data, controller=standing_controller):\\n    \\"\\"\\"Вызвать контроллер один раз и пройти 10 мс физики.\\n\\n    Пример подключения: step_simulation(model, data, controller=my_mpc).\\n    Возвращаемые моменты удерживаются между вызовами контроллера.\\n    \\"\\"\\"\\n    torque = np.asarray(controller(model, data), dtype=float)\\n    if torque.shape != (model.nu,) or not np.isfinite(torque).all():\\n        raise ValueError(f\\"Контроллер должен вернуть {model.nu} конечных моментов\\")\\n    data.ctrl[:] = np.clip(\\n        torque, model.actuator_ctrlrange[:, 0], model.actuator_ctrlrange[:, 1]\\n    )\\n    for _ in range(CONTROL_SUBSTEPS):\\n        mujoco.mj_step(model, data)\\n    # Обновить кинематику и контакты для нового состояния перед следующим MPC.\\n    mujoco.mj_forward(model, data)\\n\\n\\ndef run_viewer(model, data, duration, controller=standing_controller):\\n    # Импорт внутри функции: headless-режиму окно и GLFW не нужны.\\n    import mujoco.viewer\\n\\n    camera_id = model.camera(CAMERA_NAME).id\\n    with mujoco.viewer.launch_passive(\\n        model, data, show_left_ui=False, show_right_ui=False\\n    ) as viewer:\\n        while viewer.is_running() and data.time < duration:\\n            started = time.perf_counter()\\n            with viewer.lock():\\n                step_simulation(model, data, controller)\\n                viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FIXED\\n                viewer.cam.fixedcamid = camera_id\\n            viewer.sync()\\n            delay = model.opt.timestep * CONTROL_SUBSTEPS\\n            time.sleep(max(0.0, delay - (time.perf_counter() - started)))\\n\\n\\ndef run_headless(model, data, duration, video=None, controller=standing_controller):\\n    \\"\\"\\"Работает без окна; видео пишется по кадрам, без накопления в памяти.\\"\\"\\"\\n    if video is None:\\n        while data.time < duration - 1e-12:\\n            step_simulation(model, data, controller)\\n        return\\n\\n    import imageio.v2 as imageio\\n\\n    video = Path(video)\\n    video.parent.mkdir(parents=True, exist_ok=True)\\n    with mujoco.Renderer(model, height=HEIGHT, width=WIDTH) as renderer:\\n        with imageio.get_writer(str(video), fps=FPS, codec=\\"libx264\\") as writer:\\n            for frame in range(math.ceil(duration * FPS)):\\n                while data.time < frame / FPS - 1e-12:\\n                    step_simulation(model, data, controller)\\n                renderer.update_scene(data, camera=CAMERA_NAME)\\n                writer.append_data(renderer.render())\\n    while data.time < duration - 1e-12:\\n        step_simulation(model, data, controller)\\n    print(f\\"Видео: {video.resolve()}\\")\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--duration\\", type=float, default=10.0, help=\\"Время симуляции, с\\")\\n    parser.add_argument(\\"--headless\\", action=\\"store_true\\", help=\\"Без окна\\")\\n    parser.add_argument(\\"--video\\", type=Path, help=\\"Записать MP4 без окна\\")\\n    args = parser.parse_args()\\n    if not math.isfinite(args.duration) or args.duration <= 0:\\n        parser.error(\\"--duration должно быть положительным конечным числом\\")\\n    if not (SCENE_PATH.parent / \\"models/unitree_go2/go2.xml\\").is_file():\\n        parser.error(\\"Сначала скачайте модель: python download_model.py\\")\\n\\n    model, data = load_simulation()\\n    if args.headless or args.video:\\n        run_headless(model, data, args.duration, video=args.video)\\n    else:\\n        run_viewer(model, data, args.duration)\\n    print(f\\"t={data.time:.2f} с; высота корпуса={data.qpos[2]:.3f} м\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "go2_mpc.py": "\\"\\"\\"MPC для Go2.\\n\\nУправление робособакой при неизменном положении задних конечностей.\\nДля малых углов модель линейная, поэтому MPC решается через do-mpc. Силы переводятся в моменты\\nсуставов используя якобиан из pinnochio.\\n\\npython go2_mpc.py --shape heart \\nMUJOCO_GL=egl python go2_mpc.py --shape heart --video outputs/heart.mp4\\n\\"\\"\\"\\n\\nimport argparse\\nimport ctypes\\nimport math\\nimport sys\\nimport time\\nimport warnings\\nfrom pathlib import Path\\n\\n# ROS (например Humble) подсовывает свои eigenpy через PYTHONPATH и LD_LIBRARY_PATH,\\n# с ними pinocchio из venv не импортируется. Убираем пути ROS и заранее грузим\\n# libeigenpy из venv: так работает и из терминала, и из Jupyter.\\nsys.path[:] = [p for p in sys.path if not p.startswith(\\"/opt/ros\\")]\\n_EIGENPY = Path(sys.prefix) / \\"lib\\" / f\\"python{sys.version_info.major}.{sys.version_info.minor}\\" \\\\\\n    / \\"site-packages/cmeel.prefix/lib/libeigenpy.so\\"\\nif _EIGENPY.is_file():\\n    ctypes.CDLL(str(_EIGENPY), mode=ctypes.RTLD_GLOBAL)\\n\\nimport casadi as ca\\nif hasattr(ca.GlobalOptions, \\"setNumpyMode\\"):\\n    ca.GlobalOptions.setNumpyMode(-1)  \\nwarnings.filterwarnings(\\"ignore\\", module=\\"do_mpc\\")\\nimport do_mpc  # noqa: E402\\nimport mujoco\\nimport numpy as np\\nimport pinocchio as pin\\n\\nfrom go2_sim import (\\n    CAMERA_NAME, CONTROL_SUBSTEPS, FPS, HEIGHT, SCENE_PATH, WIDTH,\\n    load_simulation, step_simulation,\\n)\\n\\nMJCF_PATH = SCENE_PATH.parent / \\"models/unitree_go2/go2.xml\\"\\nLEGS = [\\"FL\\", \\"FR\\", \\"RL\\", \\"RR\\"]\\nFOOT_OFFSET = np.array([-0.002, 0.0, -0.213])  # центр стопы из go2.xml\\nSNOUT = np.array([0.36, 0.0, 0.0])\\n\\nDT = 0.02        \\nHORIZON = 10     \\nCOM_HEIGHT = 0.26  # желаемая высота центра масс\\nMU = 0.6         # коэффициент трения\\nFZ_MAX = 150.0   # сила на одну ногу\\nGRAVITY = np.array([0.0, 0.0, -9.81])\\n\\n\\ndef shape_point(shape, s):\\n    \\"\\"\\"По форме возвращает траекторию движения в виде (u, v)\\"\\"\\"\\n    a = 2 * math.pi * s\\n    if shape == \\"circle\\":\\n        return math.sin(a), math.cos(a)\\n    if shape == \\"heart\\":\\n        u = 16 * math.sin(a) ** 3\\n        v = 13 * math.cos(a) - 5 * math.cos(2 * a) - 2 * math.cos(3 * a) - math.cos(4 * a)\\n        return u / 17.0, (v + 2.5) / 17.0\\n    raise ValueError(shape)\\n\\n\\nclass Reference:\\n    \\"\\"\\"\\n    Рисует фигуру в зависимости от ее формы из начального положения, если находится не в начальном\\n    то робот сначала возвращается в начальное положение.\\n    \\"\\"\\"\\n    def  __init__(self, shape, period=6.0, amp_yaw=0.25, amp_pitch=0.2, t_start=1.0):\\n        self.shape, self.period, self.t_start = shape, period, t_start\\n        self.amp_yaw, self.amp_pitch = amp_yaw, amp_pitch\\n\\n    def __call__(self, t):\\n        if t < self.t_start:  \\n            k = 0.5 - 0.5 * math.cos(math.pi * max(t, 0.0) / self.t_start)\\n            u, v = shape_point(self.shape, 0.0)\\n            u, v = k * u, k * v\\n        else:\\n            u, v = shape_point(self.shape, ((t - self.t_start) / self.period) % 1.0)\\n        return -self.amp_pitch * v, self.amp_yaw * u\\n\\n    def drawing(self, t):\\n        return t >= self.t_start\\n\\n\\nclass StandingMPC:\\n    def __init__(self, model, data, reference):\\n        self.reference = reference\\n        self.calls = 0\\n        self.force = np.zeros(12)\\n\\n        self.pm = pin.buildModelFromMJCF(str(MJCF_PATH), pin.JointModelFreeFlyer())\\n        self.pm.jointPlacements[1] = pin.SE3.Identity()\\n        \\n        for leg in LEGS:\\n            jid = self.pm.getJointId(f\\"{leg}_calf_joint\\")\\n            self.pm.addFrame(pin.Frame(f\\"{leg}_foot\\", jid, 0,\\n                                       pin.SE3(np.eye(3), FOOT_OFFSET), pin.FrameType.OP_FRAME))\\n        self.pd = self.pm.createData()\\n        self.foot_ids = [self.pm.getFrameId(f\\"{leg}_foot\\") for leg in LEGS]\\n        self.act_v = [self.pm.joints[self.pm.getJointId(\\n            model.joint(model.actuator_trnid[i, 0]).name)].idx_v for i in range(model.nu)]\\n\\n        q, v = self.pin_state(data)\\n        pin.ccrba(self.pm, self.pd, q, v)           \\n        mass = self.pd.Ig.mass\\n        inertia = self.pd.Ig.inertia                 # оси\\n        com = pin.centerOfMass(self.pm, self.pd, q)\\n        pin.framesForwardKinematics(self.pm, self.pd, q)\\n        feet = [self.pd.oMf[i].translation.copy() for i in self.foot_ids]\\n        self.com0 = com.copy()\\n\\n        A = np.eye(12)\\n        A[0:3, 6:9] = DT * np.eye(3)   \\n        A[3:6, 9:12] = DT * np.eye(3)  \\n        B = np.zeros((12, 12))\\n        inv_inertia = np.linalg.inv(inertia)\\n        for i, foot in enumerate(feet):\\n            B[6:9, 3 * i:3 * i + 3] = DT * inv_inertia @ pin.skew(foot - com)\\n            B[9:12, 3 * i:3 * i + 3] = DT * np.eye(3) / mass\\n        c = np.zeros(12)\\n        c[9:12] = DT * GRAVITY\\n        self.mass, self.inertia, self.feet, self.A, self.B = mass, inertia, feet, A, B\\n\\n        # A, B выше — модель при yaw = 0 (для наглядности). В MPC учитываем поворот\\n        # корпуса по yaw: d(rpy)/dt ≈ Rz^T ω, I_мир = Rz I Rz^T. Yaw берём из\\n        # желаемой траектории (известный параметр), поэтому модель остаётся линейной.\\n        m = do_mpc.model.Model(\\"discrete\\")\\n        x = m.set_variable(\\"_x\\", \\"x\\", (12, 1))\\n        f = m.set_variable(\\"_u\\", \\"f\\", (12, 1))\\n        ref = m.set_variable(\\"_tvp\\", \\"ref\\", (12, 1))\\n        yaw = m.set_variable(\\"_tvp\\", \\"yaw\\")\\n        cz, sz = ca.cos(yaw), ca.sin(yaw)\\n        Rz = ca.vertcat(ca.horzcat(cz, -sz, 0), ca.horzcat(sz, cz, 0), ca.horzcat(0, 0, 1))\\n        moment = ca.DM(np.hstack([pin.skew(foot - com) for foot in feet])) @ f  # Σ r_i × f_i\\n        x_next = ca.DM(A) @ x + ca.DM(B) @ f + ca.DM(c)\\n        x_next[0:3] = x[0:3] + DT * Rz.T @ x[6:9]\\n        x_next[6:9] = x[6:9] + DT * Rz @ ca.DM(inv_inertia) @ Rz.T @ moment\\n        m.set_rhs(\\"x\\", x_next)\\n        Q = np.diag([200, 300, 300,  50, 50, 200,  1, 1, 1,  1, 1, 1])\\n        err = x - ref\\n        m.set_expression(\\"lterm\\", err.T @ ca.DM(Q) @ err + 1e-5 * (f.T @ f))\\n        m.set_expression(\\"mterm\\", err.T @ ca.DM(Q) @ err)\\n        m.setup()\\n\\n        mpc = do_mpc.controller.MPC(m)\\n        mpc.settings.n_horizon = HORIZON\\n        mpc.settings.t_step = DT\\n        mpc.settings.store_full_solution = False\\n        mpc.settings.supress_ipopt_output()\\n        mpc.set_objective(lterm=m.aux[\\"lterm\\"], mterm=m.aux[\\"mterm\\"])\\n        mpc.set_rterm(f=1e-5)\\n\\n        lower = np.tile([-np.inf, -np.inf, 0.0], 4)\\n        upper = np.tile([np.inf, np.inf, FZ_MAX], 4)\\n        mpc.bounds[\\"lower\\", \\"_u\\", \\"f\\"] = lower\\n        mpc.bounds[\\"upper\\", \\"_u\\", \\"f\\"] = upper\\n        fx, fy, fz = f[0::3], f[1::3], f[2::3]  \\n        mpc.set_nl_cons(\\"friction\\", ca.vertcat(fx - MU * fz, -fx - MU * fz,\\n                                               fy - MU * fz, -fy - MU * fz), ub=0)\\n\\n        tvp = mpc.get_tvp_template()\\n\\n        def tvp_fun(t_now):\\n            t_now = float(np.squeeze(t_now))  # do-mpc передаёт время массивом\\n            for k in range(HORIZON + 1):\\n                pitch, yaw = self.reference(t_now + k * DT)\\n                tvp[\\"_tvp\\", k, \\"ref\\"] = np.r_[0.0, pitch, yaw,\\n                                              self.com0[0], self.com0[1], COM_HEIGHT,\\n                                              np.zeros(6)]\\n                tvp[\\"_tvp\\", k, \\"yaw\\"] = yaw\\n            return tvp\\n\\n        mpc.set_tvp_fun(tvp_fun)\\n        mpc.setup()\\n        mpc.x0 = self.measure(data)\\n        mpc.set_initial_guess()\\n        self.mpc = mpc\\n\\n    def pin_state(self, data):\\n        \\"\\"\\"Перевод форматов Mujoco в Pinochio\\"\\"\\"\\n        qpos, qvel = data.qpos, data.qvel\\n        q = np.r_[qpos[0:3], qpos[4:7], qpos[3], qpos[7:]]\\n        R = pin.Quaternion(qpos[3], *qpos[4:7]).toRotationMatrix()\\n        v = np.r_[R.T @ qvel[0:3], qvel[3:6], qvel[6:]]  # в pinocchio лин. скорость локальная\\n        return q, v\\n\\n    def measure(self, data):\\n        \\"\\"\\"Измерение состояния модели\\"\\"\\"\\n        q, v = self.pin_state(data)\\n        R = pin.Quaternion(data.qpos[3], *data.qpos[4:7]).toRotationMatrix()\\n        com = pin.centerOfMass(self.pm, self.pd, q, v)\\n        return np.r_[pin.rpy.matrixToRpy(R), com, R @ data.qvel[3:6], self.pd.vcom[0]]\\n\\n    def __call__(self, model, data):\\n        if self.calls % round(DT / (model.opt.timestep * CONTROL_SUBSTEPS)) == 0:\\n            self.force = np.asarray(self.mpc.make_step(self.measure(data))).ravel()\\n        self.calls += 1\\n\\n        q, _ = self.pin_state(data)\\n        pin.computeJointJacobians(self.pm, self.pd, q)\\n        pin.framesForwardKinematics(self.pm, self.pd, q)\\n        tau_v = np.zeros(self.pm.nv)\\n        for i, fid in enumerate(self.foot_ids):\\n            J = pin.getFrameJacobian(self.pm, self.pd, fid, pin.LOCAL_WORLD_ALIGNED)[:3]\\n            tau_v -= J.T @ self.force[3 * i:3 * i + 3]\\n        return tau_v[self.act_v]\\n\\n\\ndef snout_position(data):\\n    return data.body(\\"base\\").xpos + data.body(\\"base\\").xmat.reshape(3, 3) @ SNOUT\\n\\n\\ndef draw_trail(scene, trail):\\n    for p in trail:\\n        if scene.ngeom >= scene.maxgeom:\\n            break\\n        mujoco.mjv_initGeom(scene.geoms[scene.ngeom], mujoco.mjtGeom.mjGEOM_SPHERE,\\n                            np.array([0.004, 0, 0]), p, np.eye(3).ravel(),\\n                            np.array([1.0, 0.2, 0.2, 1.0], dtype=np.float32))\\n        scene.ngeom += 1\\n\\n\\ndef run(shape, duration, video=None):\\n    model, data = load_simulation()\\n    reference = Reference(shape)\\n    controller = StandingMPC(model, data, reference)\\n    trail = []\\n\\n    def step():\\n        step_simulation(model, data, controller)\\n        if reference.drawing(data.time) and data.time < reference.t_start + reference.period:\\n            trail.append(snout_position(data).copy())\\n\\n    if video is None:\\n        from mujoco import viewer as mj_viewer  # окну нужен GLFW, импорт только здесь\\n        with mj_viewer.launch_passive(model, data, show_left_ui=False,\\n                                          show_right_ui=False) as viewer:\\n            viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FIXED\\n            viewer.cam.fixedcamid = model.camera(CAMERA_NAME).id\\n            while viewer.is_running() and data.time < duration:\\n                started = time.perf_counter()\\n                with viewer.lock():\\n                    step()\\n                    viewer.user_scn.ngeom = 0\\n                    draw_trail(viewer.user_scn, trail)\\n                viewer.sync()\\n                delay = model.opt.timestep * CONTROL_SUBSTEPS\\n                time.sleep(max(0.0, delay - (time.perf_counter() - started)))\\n    else:\\n        import imageio.v2 as imageio\\n        video = Path(video)\\n        video.parent.mkdir(parents=True, exist_ok=True)\\n        with mujoco.Renderer(model, height=HEIGHT, width=WIDTH) as renderer, \\\\\\n                imageio.get_writer(str(video), fps=FPS, codec=\\"libx264\\") as writer:\\n            for frame in range(math.ceil(duration * FPS)):\\n                while data.time < frame / FPS - 1e-12:\\n                    step()\\n                renderer.update_scene(data, camera=CAMERA_NAME)\\n                draw_trail(renderer.scene, trail)\\n                writer.append_data(renderer.render())\\n        print(f\\"Видео: {video.resolve()}\\")\\n    rpy = pin.rpy.matrixToRpy(data.body(\\"base\\").xmat.reshape(3, 3))\\n    print(f\\"t={data.time:.2f} с; rpy={np.round(rpy, 3)}\\")\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--shape\\", choices=[\\"circle\\", \\"heart\\"], default=\\"heart\\")\\n    parser.add_argument(\\"--duration\\", type=float, default=8.0)\\n    parser.add_argument(\\"--video\\", type=Path, help=\\"Записать MP4 вместо окна\\")\\n    args = parser.parse_args()\\n    run(args.shape, args.duration, args.video)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "scene.xml": "<mujoco model=\\"Unitree Go2 — fixed camera\\">\\n  <include file=\\"models/unitree_go2/go2.xml\\"/>\\n  <compiler meshdir=\\"models/unitree_go2/assets\\"/>\\n  <option timestep=\\"0.002\\" integrator=\\"implicitfast\\"/>\\n\\n  <visual>\\n    <global offwidth=\\"960\\" offheight=\\"544\\"/>\\n    <headlight ambient=\\"0.35 0.35 0.35\\" diffuse=\\"0.7 0.7 0.7\\"/>\\n  </visual>\\n  <asset>\\n    <texture name=\\"sky\\" type=\\"skybox\\" builtin=\\"gradient\\" width=\\"512\\" height=\\"3072\\"\\n             rgb1=\\"0.28 0.34 0.40\\" rgb2=\\"0.07 0.10 0.14\\"/>\\n    <texture name=\\"ground\\" type=\\"2d\\" builtin=\\"checker\\" width=\\"512\\" height=\\"512\\"\\n             rgb1=\\"0.24 0.27 0.30\\" rgb2=\\"0.32 0.35 0.38\\"/>\\n    <material name=\\"ground\\" texture=\\"ground\\" texrepeat=\\"8 8\\" texuniform=\\"true\\"/>\\n  </asset>\\n\\n  <worldbody>\\n    <light pos=\\"0 -2 3\\" dir=\\"0 0 -1\\" directional=\\"true\\"/>\\n    <geom name=\\"floor\\" type=\\"plane\\" size=\\"0 0 0.1\\" material=\\"ground\\"/>\\n\\n    <!-- Камера перед роботом по оси +X, высота 35 см, вид на лицо.\\n         Она не прикреплена к роботу и не отслеживает его движение. -->\\n    <camera name=\\"fixed\\" mode=\\"fixed\\" pos=\\"1.2 0 0.35\\"\\n            xyaxes=\\"0 1 0 -0.099504 0 0.995037\\" fovy=\\"40\\"/>\\n  </worldbody>\\n</mujoco>\\n", "download_model.py": "\\"\\"\\"Скачать только Unitree Go2 из официального MuJoCo Menagerie.\\"\\"\\"\\n\\nfrom pathlib import Path\\nimport urllib.request\\nimport xml.etree.ElementTree as ET\\n\\n\\n# Фиксированная версия модели: повторный запуск не изменит физику робота.\\nREVISION = \\"f3b6013629c7b94f8ee93fd977b161fa8350f9bc\\"\\nBASE_URL = (\\n    \\"https://raw.githubusercontent.com/google-deepmind/\\"\\n    f\\"mujoco_menagerie/{REVISION}/unitree_go2\\"\\n)\\nMODEL_DIR = Path(__file__).resolve().parent / \\"models\\" / \\"unitree_go2\\"\\n\\n\\ndef download(relative_path):\\n    destination = MODEL_DIR / relative_path\\n    if destination.is_file():\\n        return\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    print(f\\"Скачивание {relative_path}\\", flush=True)\\n    request = urllib.request.Request(\\n        f\\"{BASE_URL}/{relative_path}\\", headers={\\"User-Agent\\": \\"go2-mujoco-demo\\"}\\n    )\\n    with urllib.request.urlopen(request, timeout=60) as response:\\n        content = response.read()\\n    # Незавершённая загрузка не оставляет файл с именем готового ресурса.\\n    temporary = destination.with_suffix(destination.suffix + \\".part\\")\\n    temporary.write_bytes(content)\\n    temporary.replace(destination)\\n\\n\\ndef main():\\n    for filename in (\\"go2.xml\\", \\"LICENSE\\", \\"README.md\\"):\\n        download(filename)\\n    root = ET.parse(MODEL_DIR / \\"go2.xml\\").getroot()\\n    for mesh in root.findall(\\"./asset/mesh\\"):\\n        download(\\"assets/\\" + mesh.attrib[\\"file\\"])\\n    print(f\\"Модель готова: {MODEL_DIR}\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "lab_helpers.py": "import math\\nfrom pathlib import Path\\nimport numpy as np\\nimport mujoco\\nfrom scipy.spatial import Delaunay\\nfrom go2_sim import load_simulation, step_simulation, CAMERA_NAME\\nfrom go2_mpc import StandingMPC, snout_position\\n\\nW,H = 640,360\\nRAMP=0.8\\nSETTLE=1.8\\nSAMPLE=0.3\\n\\n\\ndef nose_pixel(model,data):\\n    cid=model.camera(CAMERA_NAME).id\\n    camera=data.cam_xmat[cid].reshape(3,3)\\n    point=camera.T @ (snout_position(data)-data.cam_xpos[cid])\\n    depth=-point[2]\\n    if depth<=0: raise RuntimeError(\'Нос находится позади камеры\')\\n    f=H/(2*np.tan(np.deg2rad(model.cam_fovy[cid])/2))\\n    return np.array([W/2+f*point[0]/depth,H/2-f*point[1]/depth])\\n\\n\\nclass Experiment:\\n    def __init__(self):\\n        self.model,self.data=load_simulation()\\n        self.controller=StandingMPC(self.model,self.data,lambda t:(0.,0.))\\n\\n    def reset(self,reference):\\n        model,data=self.model,self.data\\n        mujoco.mj_resetDataKeyframe(model,data,model.key(\'home\').id)\\n        mujoco.mj_forward(model,data)\\n        self.controller.reference=reference\\n        self.controller.calls=0\\n        self.controller.force[:]=0\\n        self.controller.mpc.reset_history()\\n        self.controller.mpc.x0=self.controller.measure(data)\\n        self.controller.mpc.u0=np.zeros(12)\\n        self.controller.mpc.set_initial_guess()\\n        # Версия do-mpc зафиксирована: следующий solve не использует прежние множители.\\n        self.controller.mpc.flags[\'initial_run\']=False\\n\\n    def step(self):\\n        step_simulation(self.model,self.data,self.controller)\\n        if not np.all(np.isfinite(self.data.qpos)) or self.data.qpos[2]<.17:\\n            raise RuntimeError(\'Робот потерял стойку: уменьшите диапазон команд\')\\n\\n    def measure(self,pitch,yaw):\\n        def reference(t):\\n            blend=.5-.5*np.cos(np.pi*np.clip(t/RAMP,0,1))\\n            return float(blend*pitch),float(blend*yaw)\\n        self.reset(reference)\\n        pixels=[];states=[]\\n        while self.data.time<SETTLE+SAMPLE-1e-9:\\n            self.step()\\n            if self.data.time>=SETTLE:\\n                pixels.append(nose_pixel(self.model,self.data))\\n                states.append(self.controller.measure(self.data))\\n        pixels=np.array(pixels);states=np.array(states)\\n        if np.max(np.linalg.norm(states[:,-3:],axis=1))>.025:\\n            raise RuntimeError(\'Измерение ещё не установилось: увеличьте SETTLE\')\\n        return dict(command=np.array([pitch,yaw]),pixel=pixels.mean(axis=0),\\n                    scatter=pixels.std(axis=0),rpy=states[:,:3].mean(axis=0),\\n                    height=float(self.data.qpos[2]))\\n\\n\\ndef sample_commands(experiment,commands,verbose=True):\\n    records=[]\\n    for i,(pitch,yaw) in enumerate(commands):\\n        records.append(experiment.measure(pitch,yaw))\\n        if verbose and ((i+1)%5==0 or i+1==len(commands)):\\n            print(f\'{i+1}/{len(commands)}\',flush=True)\\n    return {key:np.array([record[key] for record in records]) for key in records[0]}\\n", "drawing_helpers.py": "import math\\nfrom pathlib import Path\\nimport numpy as np\\nimport mujoco\\nimport imageio.v2 as imageio\\nfrom PIL import Image, ImageDraw\\nfrom scipy.spatial import Delaunay\\nfrom go2_sim import CAMERA_NAME\\nfrom go2_mpc import snout_position,draw_trail\\nfrom lab_helpers import W,H,nose_pixel\\n\\n\\ndef checked_inverse(calibration, interpolator):\\n    \\"\\"\\"Проверить область запросов и диапазон команд готового интерполянта.\\"\\"\\"\\n    hull=Delaunay(calibration[\'pixel\'])\\n    lower=calibration[\'command\'].min(axis=0)\\n    upper=calibration[\'command\'].max(axis=0)\\n    def inverse(points):\\n        points=np.asarray(points,dtype=float).reshape(-1,2)\\n        if not np.all(np.isfinite(points)) or np.any(hull.find_simplex(points)<0):\\n            raise ValueError(\'Траектория выходит за калиброванную область\')\\n        result=np.asarray(interpolator(points),dtype=float)\\n        if result.shape!=points.shape or not np.all(np.isfinite(result)):\\n            raise ValueError(\'Интерполятор должен вернуть конечный массив Nx2\')\\n        if np.any(result<lower-1e-6) or np.any(result>upper+1e-6):\\n            raise ValueError(\'Предсказанные команды вышли за калибровочный диапазон\')\\n        return result\\n    return inverse\\n\\n\\ndef screen_path(calibration,shape=\'heart\',scale=.55,points=241):\\n    phase=np.linspace(0,2*np.pi,points)\\n    if shape==\'heart\':\\n        u=16*np.sin(phase)**3/17\\n        v=-(13*np.cos(phase)-5*np.cos(2*phase)-2*np.cos(3*phase)-np.cos(4*phase)+2.5)/17\\n    elif shape==\'circle\':\\n        u,v=np.cos(phase),np.sin(phase)\\n    else:\\n        raise ValueError(\'Используйте heart/circle или передайте свою траекторию Nx2\')\\n    center=calibration[\'pixel\'].mean(axis=0)\\n    half_range=np.ptp(calibration[\'pixel\'],axis=0)/2\\n    return center+scale*np.column_stack([u,v])*half_range\\n\\n\\ndef draw_frame(experiment,renderer,target_path=None,trail=None):\\n    renderer.update_scene(experiment.data,camera=CAMERA_NAME)\\n    draw_trail(renderer.scene,[snout_position(experiment.data)])\\n    frame=Image.fromarray(renderer.render())\\n    painter=ImageDraw.Draw(frame)\\n    if target_path is not None:\\n        painter.line([tuple(p) for p in target_path],fill=(0,230,255),width=2)\\n    if trail is not None and len(trail)>1:\\n        painter.line([tuple(p) for p in trail],fill=(255,70,70),width=2)\\n    return np.asarray(frame)\\n\\n\\ndef follow_screen_path(experiment,target_path,inverse,period=16.,draw_start=2.1,\\n                        video_path=\'go2_drawing.mp4\',fps=15):\\n    target_path=np.asarray(target_path,dtype=float)\\n    if target_path.ndim!=2 or target_path.shape[1]!=2 or len(target_path)<2:\\n        raise ValueError(\'Траектория должна иметь форму Nx2, N>=2\')\\n    commands=inverse(target_path)\\n    times=np.linspace(0,period,len(target_path))\\n    def reference(t):\\n        if t<draw_start:\\n            blend=.5-.5*np.cos(np.pi*np.clip(t/.8,0,1))\\n            return commands[0]*blend\\n        return np.array([np.interp(t-draw_start,times,commands[:,j]) for j in range(2)])\\n    experiment.reset(reference)\\n    duration=draw_start+period\\n    log_time=[];log_pixels=[];log_target=[];log_command=[]\\n    video_path=Path(video_path);video_path.parent.mkdir(parents=True,exist_ok=True)\\n    # Запись идёт покадрово: кадры не накапливаются в памяти.\\n    with mujoco.Renderer(experiment.model,height=H,width=W) as renderer, \\\\\\n         imageio.get_writer(video_path,fps=fps,codec=\'libx264\',macro_block_size=1) as writer:\\n        for frame_time in np.arange(math.ceil(duration*fps))/fps:\\n            while experiment.data.time<frame_time-1e-10:\\n                experiment.step()\\n                t=experiment.data.time\\n                if draw_start<=t<=duration:\\n                    goal=np.array([np.interp(t-draw_start,times,target_path[:,j]) for j in range(2)])\\n                    log_time.append(t-draw_start)\\n                    log_pixels.append(nose_pixel(experiment.model,experiment.data))\\n                    log_target.append(goal)\\n                    log_command.append(reference(t))\\n            writer.append_data(draw_frame(experiment,renderer,target_path,log_pixels))\\n        while experiment.data.time<duration-1e-10:\\n            experiment.step()\\n            t=experiment.data.time\\n            if t>=draw_start:\\n                goal=np.array([np.interp(t-draw_start,times,target_path[:,j]) for j in range(2)])\\n                log_time.append(t-draw_start);log_pixels.append(nose_pixel(experiment.model,experiment.data))\\n                log_target.append(goal);log_command.append(reference(t))\\n    return dict(t=np.array(log_time),pixel=np.array(log_pixels),target=np.array(log_target),\\n                command=np.array(log_command),video_path=str(video_path))\\n"}')
for name, content in PROJECT_FILES.items():
    (WORK_DIR / name).write_text(content, encoding="utf-8")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
subprocess.run([sys.executable, str(WORK_DIR / "download_model.py")], check=True)
from lab_helpers import Experiment, sample_commands, nose_pixel, W, H
from drawing_helpers import checked_inverse, screen_path, draw_frame, follow_screen_path
experiment = Experiment()
neutral = experiment.measure(0., 0.)
with mujoco.Renderer(experiment.model, height=H, width=W) as renderer:
    neutral_image = draw_frame(experiment, renderer)


### Г1. Сбор соответствий между командами и пикселями

Запустите готовую ячейку сбора данных с `grid_size = 5`. Она переберёт 25 сочетаний углов: по пять значений `pitch` и `yaw`.

Для каждого сочетания робот начинает из одинакового положения, плавно меняет наклон и поворот, затем ожидает установления позы. Положение метки измеряется несколько раз и усредняется. Это модель измерения: распознавать метку по изображению самостоятельно не нужно.

Результат записан в `calibration`:

- `calibration["command"]` — таблица команд; в каждой строке `[pitch, yaw]`.
- `calibration["pixel"]` — таблица наблюдаемых точек; в соответствующей строке `[u, v]`.

Посмотрите на график собранных точек. Равномерным изменениям углов не обязательно соответствуют равномерные смещения на экране. Камеру и параметры стойки после сбора данных не меняйте: иначе полученные соответствия перестанут описывать эксперимент.


In [ ]:
grid_size = 3 #@param [3, 5, 7] {type:"raw"}
pitch_axis = np.linspace(-0.18, 0.18, grid_size)
yaw_axis = np.linspace(-0.22, 0.22, grid_size)
calibration_commands = np.array([(pitch, yaw) for pitch in pitch_axis for yaw in yaw_axis])

started = time.perf_counter()
calibration = sample_commands(experiment, calibration_commands)
print(f"Время калибровки: {time.perf_counter()-started:.1f} с")
print("Максимальный разброс координаты:", calibration["scatter"].max(), "пикселя")
np.savez(WORK_DIR / "calibration.npz", **calibration)


In [ ]:
uv_grid = calibration["pixel"].reshape(grid_size, grid_size, 2)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(neutral_image)
for row in uv_grid:
    axes[0].plot(row[:, 0], row[:, 1], "o-", color="cyan", ms=3)
for column in uv_grid.transpose(1, 0, 2):
    axes[0].plot(column[:, 0], column[:, 1], "-", color="yellow", lw=1)
axes[0].set(title="Измеренная сетка в изображении", xlabel="u, пиксели", ylabel="v, пиксели")

for row in uv_grid:
    axes[1].plot(row[:, 0], row[:, 1], "-", color="0.7")
for column in uv_grid.transpose(1, 0, 2):
    axes[1].plot(column[:, 0], column[:, 1], "-", color="0.7")
points = axes[1].scatter(calibration["pixel"][:, 0], calibration["pixel"][:, 1],
                          c=calibration["command"][:, 0], cmap="coolwarm", s=45)
fig.colorbar(points, ax=axes[1], label="Команда pitch, рад")
axes[1].invert_yaxis()
axes[1].set_aspect("equal")
axes[1].set(title="Увеличенный участок", xlabel="u, пиксели", ylabel="v, пиксели")
plt.tight_layout()
plt.show()


### Г2. Построение обратной зависимости

Заполните `make_inverse(pixels, commands)` с помощью готовой функции из `scipy.interpolate`. Подойдут, например, `LinearNDInterpolator` или `CloughTocher2DInterpolator`: они работают с точками, которые не обязаны лежать на прямоугольной сетке.

Здесь `pixels` — таблица входных точек `[u, v]`, а `commands` — таблица известных ответов `[pitch, yaw]`. Их строки соответствуют друг другу. Верните функцию, которая принимает таблицу новых экранных точек и возвращает таблицу команд. Можно построить отдельную интерполяцию для каждого угла либо передать оба столбца значений одному интерполятору.

Пример того, как будет использоваться результат:

```python
inverse = make_inverse(calibration["pixel"], calibration["command"])
commands_for_points = inverse(calibration["pixel"][:3])
# Три строки входа -> три строки выхода; в выходе столбцы pitch и yaw.
```

Готовая обёртка в следующей ячейке проверяет, что запрашиваемые точки находятся внутри области, охваченной калибровкой. За её пределами результат в этой работе не ищем.

Затем запустите проверочный опыт. Он подаст **новые** команды, измерит положения метки, восстановит команды по вашему интерполятору и снова выполнит их на роботе. Сравните исходные и восстановленные углы, а также желаемые и достигнутые точки. Эти данные не добавляйте в калибровочную таблицу: иначе проверка перестанет быть независимой.


In [ ]:
def make_inverse(pixels, commands):
    # TODO Г2: выбрать и вызвать готовую функцию scipy.interpolate.
    # Вернуть вызываемый интерполятор (m,2) -> (m,2).
    raise NotImplementedError("Г2: обратная интерполяция средствами SciPy")


interpolator = make_inverse(calibration["pixel"], calibration["command"])
inverse_commands = checked_inverse(calibration, interpolator)


In [ ]:
rng = np.random.default_rng(17)
validation_commands = rng.uniform([-0.13, -0.16], [0.13, 0.16], size=(8, 2))
validation = sample_commands(experiment, validation_commands)
predicted_commands = inverse_commands(validation["pixel"])
achieved = sample_commands(experiment, predicted_commands)

angle_error = predicted_commands-validation["command"]
pixel_error = np.linalg.norm(achieved["pixel"]-validation["pixel"], axis=1)
print("RMSE каждой команды, рад:", np.sqrt(np.mean(angle_error**2, axis=0)))
print("RMSE положения, пиксели:", np.sqrt(np.mean(pixel_error**2)))
print("Максимальная ошибка положения, пиксели:", pixel_error.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(validation["pixel"][:, 0], validation["pixel"][:, 1], label="Цели")
axes[0].scatter(achieved["pixel"][:, 0], achieved["pixel"][:, 1], marker="x", label="Достигнутые точки")
for target, actual in zip(validation["pixel"], achieved["pixel"]):
    axes[0].plot([target[0], actual[0]], [target[1], actual[1]], color="0.5")
axes[0].invert_yaxis()
axes[0].set_aspect("equal")
axes[0].set(xlabel="u, пиксели", ylabel="v, пиксели")
axes[0].legend()
axes[1].bar(np.arange(1, len(pixel_error)+1), pixel_error)
axes[1].set(xlabel="Номер проверочной точки", ylabel="Ошибка, пиксели")
plt.tight_layout()
plt.show()


### Г3. Рисунок по экранным точкам

Сначала запустите готовый пример с сердцем или окружностью. Массив `target_pixels` содержит точки рисунка: каждая строка — `[u, v]`. Интерполятор переводит их в команды, а готовый код последовательно передаёт команды роботу.

Затем замените `target_pixels` своей фигурой. Например, задайте две координаты через один параметр и соберите их в таблицу с помощью `np.column_stack`. Точки должны идти в порядке обхода рисунка, находиться внутри калиброванной области и не содержать резких скачков. Для замкнутого рисунка последняя точка совпадает с первой.

На видео голубым показана желаемая линия, красным — фактический след. Параметр `drawing_period` задаёт время обхода в секундах. При быстром движении робот может не успевать принимать позы, по которым собиралась калибровка.


In [ ]:
shape = "heart" #@param ["heart", "circle"]
figure_scale = 0.55 #@param {type:"slider", min:0.2, max:0.85, step:0.05}
target_pixels = screen_path(calibration, shape=shape, scale=figure_scale)

# TODO: замените target_pixels своей траекторией формы (N, 2).
# В каждой строке: [u, v], пиксели. Для замкнутой фигуры конец должен совпадать с началом.
target_commands = inverse_commands(target_pixels)

plt.figure(figsize=(6, 5))
plt.scatter(calibration["pixel"][:, 0], calibration["pixel"][:, 1], s=12, color="0.6",
             label="Калибровочные точки")
plt.plot(target_pixels[:, 0], target_pixels[:, 1], color="tab:blue", label="Заданная фигура")
plt.gca().invert_yaxis()
plt.gca().set_aspect("equal")
plt.xlabel("u, пиксели")
plt.ylabel("v, пиксели")
plt.legend()
plt.show()


In [ ]:
drawing_period = 16.0 #@param {type:"slider", min:6, max:24, step:2}
drawing_result = follow_screen_path(
    experiment, target_pixels, inverse_commands, period=drawing_period,
    video_path=WORK_DIR / "drawing.mp4")
display(Video(drawing_result["video_path"], embed=True, width=720))

tracking_error = np.linalg.norm(drawing_result["pixel"]-drawing_result["target"], axis=1)
print("RMSE слежения, пиксели:", np.sqrt(np.mean(tracking_error**2)))
print("Максимальная ошибка слежения, пиксели:", tracking_error.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(drawing_result["target"][:, 0], drawing_result["target"][:, 1],
             "--", label="Задание")
axes[0].plot(drawing_result["pixel"][:, 0], drawing_result["pixel"][:, 1], label="След")
axes[0].invert_yaxis()
axes[0].set_aspect("equal")
axes[0].set(xlabel="u, пиксели", ylabel="v, пиксели")
axes[0].legend()
axes[1].plot(drawing_result["t"], tracking_error)
axes[1].set(xlabel="Время рисования, с", ylabel="Ошибка, пиксели")
plt.tight_layout()
plt.show()


### Сравнение результатов

1. Постройте обратную зависимость двумя способами из SciPy. Сравните ошибки на одном и том же наборе проверочных точек из Г2. Какой способ лучше в этом опыте?
2. Запустите свою фигуру с двумя значениями `drawing_period`. Сравните следы и ошибки. Почему изменение времени обхода влияет на результат, хотя интерполятор не менялся?
3. Объясните, почему после перемещения камеры нужно заново собирать калибровочные данные.

**Дополнительно.** Повторите калибровку с сеткой 3×3. Сравните её с сеткой 5×5 на одинаковых проверочных точках. Во всех опытах сохраните положение камеры и диапазоны углов.

**Выводы Г.** Укажите выбранный интерполятор и полученные ошибки. Обсудите отдельно попадание в неподвижные точки и следование движущейся цели.

Ваш ответ: …
